## AQI Prediction for Indian Cities: Preprocessing and Feature Engineering

## 1. Load the data

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("Data/city_day.csv", parse_dates=["Date"])
df = df.sort_values(["City", "Date"]).reset_index(drop=True)
print(df.shape)

(29531, 16)


## 2. Preprocessing

In [2]:
df["AQI_tomorrow"] = df.groupby("City")["AQI"].shift(-1)
df["next_date"] = df.groupby("City")["Date"].shift(-1)
df.loc[(df["next_date"] - df["Date"]).dt.days != 1, "AQI_tomorrow"] = np.nan

print(df[["City", "Date", "AQI", "AQI_tomorrow"]].head(10))
print("Rows with a valid tomorrow target:", df["AQI_tomorrow"].notna().sum())

        City       Date  AQI  AQI_tomorrow
0  Ahmedabad 2015-01-01  NaN           NaN
1  Ahmedabad 2015-01-02  NaN           NaN
2  Ahmedabad 2015-01-03  NaN           NaN
3  Ahmedabad 2015-01-04  NaN           NaN
4  Ahmedabad 2015-01-05  NaN           NaN
5  Ahmedabad 2015-01-06  NaN           NaN
6  Ahmedabad 2015-01-07  NaN           NaN
7  Ahmedabad 2015-01-08  NaN           NaN
8  Ahmedabad 2015-01-09  NaN           NaN
9  Ahmedabad 2015-01-10  NaN           NaN
Rows with a valid tomorrow target: 24849


### 2.1 Creating tomorrow's AQI as the target

In [3]:
df["AQI_tomorrow"] = df.groupby("City")["AQI"].shift(-1)
df["next_date"] = df.groupby("City")["Date"].shift(-1)
df.loc[(df["next_date"] - df["Date"]).dt.days != 1, "AQI_tomorrow"] = np.nan

print(df[["City", "Date", "AQI", "AQI_tomorrow"]].head(10))
print("Rows with a valid tomorrow target:", df["AQI_tomorrow"].notna().sum())

        City       Date  AQI  AQI_tomorrow
0  Ahmedabad 2015-01-01  NaN           NaN
1  Ahmedabad 2015-01-02  NaN           NaN
2  Ahmedabad 2015-01-03  NaN           NaN
3  Ahmedabad 2015-01-04  NaN           NaN
4  Ahmedabad 2015-01-05  NaN           NaN
5  Ahmedabad 2015-01-06  NaN           NaN
6  Ahmedabad 2015-01-07  NaN           NaN
7  Ahmedabad 2015-01-08  NaN           NaN
8  Ahmedabad 2015-01-09  NaN           NaN
9  Ahmedabad 2015-01-10  NaN           NaN
Rows with a valid tomorrow target: 24849


### 2.2 Removing copied-CO rows and capping AQI at 500

In [4]:
# Drop the Ahmedabad rows where NO equals CO (copied CO column)
copied = (df["City"] == "Ahmedabad") & (df["NO"] == df["CO"])
print("Rows dropped (copied CO):", copied.sum())
df = df[~copied].copy()

# Cap AQI at the official scale maximum of 500
df["AQI"] = df["AQI"].clip(upper=500)
df["AQI_tomorrow"] = df["AQI_tomorrow"].clip(upper=500)

print(df.shape)
print(df["AQI"].describe())

Rows dropped (copied CO): 1317
(28214, 18)
count    23604.000000
mean       149.720979
std        100.686049
min         13.000000
25%         79.000000
50%        113.000000
75%        187.000000
max        500.000000
Name: AQI, dtype: float64


### 2.3 Splitting into train and test by date

In [5]:
# Drop rows with no target, then split by date (no random shuffling)
model_df = df.dropna(subset=["AQI_tomorrow"]).copy()

cutoff = pd.Timestamp("2019-01-01")
train = model_df[model_df["Date"] < cutoff]
test = model_df[model_df["Date"] >= cutoff]

print("Train rows:", len(train), "| Test rows:", len(test))
print("Train dates:", train["Date"].min().date(), "to", train["Date"].max().date())
print("Test dates:", test["Date"].min().date(), "to", test["Date"].max().date())

Train rows: 12575 | Test rows: 11027
Train dates: 2015-01-01 to 2018-12-31
Test dates: 2019-01-01 to 2020-06-30


### 2.4 Filling missing pollutant values using training data only

In [6]:
features = ["PM2.5", "PM10", "NO2", "SO2", "CO", "O3", "NH3"]

# Learn the fill values from the training set only
fill_values = train[features].median()
print(fill_values)

# Apply the same values to train and test
train = train.copy()
test = test.copy()
train[features] = train[features].fillna(fill_values)
test[features] = test[features].fillna(fill_values)

print("Missing after filling (train):", train[features].isna().sum().sum())
print("Missing after filling (test):", test[features].isna().sum().sum())

PM2.5     57.150
PM10     108.635
NO2       24.975
SO2        7.970
CO         0.970
O3        31.210
NH3       21.180
dtype: float64
Missing after filling (train): 0
Missing after filling (test): 0


### 2.5 Adding date features (month, day of week, season)

In [7]:
def add_date_features(d):
    d = d.copy()
    d["month"] = d["Date"].dt.month
    d["day_of_week"] = d["Date"].dt.dayofweek
    d["season"] = d["month"].map({
        12: "winter", 1: "winter", 2: "winter",
        3: "summer", 4: "summer", 5: "summer",
        6: "monsoon", 7: "monsoon", 8: "monsoon", 9: "monsoon",
        10: "post_monsoon", 11: "post_monsoon",
    })
    return d

train = add_date_features(train)
test = add_date_features(test)

print(train[["Date", "month", "day_of_week", "season"]].head())
print(test["season"].value_counts())

           Date  month  day_of_week        season
1013 2017-10-10     10            1  post_monsoon
1023 2017-10-20     10            4  post_monsoon
1024 2017-10-21     10            5  post_monsoon
1025 2017-10-22     10            6  post_monsoon
1026 2017-10-23     10            0  post_monsoon
season
summer          3803
winter          3023
monsoon         2941
post_monsoon    1260
Name: count, dtype: int64


### 2.6 Selecting the final features and target

In [8]:
# One-hot encode season (text to numbers), fitted on column names only
train = pd.get_dummies(train, columns=["season"], dtype=int)
test = pd.get_dummies(test, columns=["season"], dtype=int)

season_cols = [c for c in train.columns if c.startswith("season_")]
test = test.reindex(columns=train.columns, fill_value=0)

feature_cols = ["PM2.5", "PM10", "NO2", "SO2", "CO", "O3", "NH3", "month", "day_of_week"] + season_cols

X_train, y_train = train[feature_cols], train["AQI_tomorrow"]
X_test, y_test = test[feature_cols], test["AQI_tomorrow"]

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Features:", feature_cols)

X_train: (12575, 13) | X_test: (11027, 13)
Features: ['PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3', 'NH3', 'month', 'day_of_week', 'season_monsoon', 'season_post_monsoon', 'season_summer', 'season_winter']


## 3. Baseline model

In [9]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
pred = baseline.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print(f"Baseline MAE:  {mae:.2f}")
print(f"Baseline RMSE: {rmse:.2f}")
print(f"Baseline R2:   {r2:.3f}")

Baseline MAE:  82.16
Baseline RMSE: 93.71
Baseline R2:   -0.301


## 4. Model comparison

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred),
    })

results_df = pd.DataFrame(results).round(3)
print(results_df.sort_values("MAE"))

               Model     MAE    RMSE     R2
2  Gradient Boosting  19.277  28.705  0.878
1      Random Forest  19.408  29.044  0.875
0  Linear Regression  26.186  35.011  0.818


## 5. Save the best model

In [12]:
import joblib

best_name = results_df.sort_values("MAE").iloc[0]["Model"]
best_model = models[best_name]

joblib.dump(best_model, "models/best_model.pkl")
print("Saved:", best_name)

Saved: Gradient Boosting
